# Used Car Market - Exploratory Data Analysis

Load the cleaned dataset, explore price, vehicle, and seller patterns, and
export charts to `outputs/charts/`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = ROOT / "data" / "processed" / "used_cars_cleaned.csv"
CHART_DIR = ROOT / "outputs" / "charts"
CHART_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")
df = pd.read_csv(DATA_PATH)
df.shape

In [ ]:
df.info()
df.describe(include="all").T

## Market overview

Distribution of listing prices and mileage.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df["price"], bins=40, ax=axes[0])
axes[0].set_title("Price distribution")
sns.histplot(df["mileage"], bins=40, ax=axes[1], color="darkorange")
axes[1].set_title("Mileage distribution")
fig.tight_layout()
fig.savefig(CHART_DIR / "market_overview.png", dpi=150)
plt.show()

## Pricing analysis

Average price by brand and by model year.

In [ ]:
top_brands = df["make"].value_counts().head(15).index
brand_price = (
    df[df["make"].isin(top_brands)]
    .groupby("make")["price"]
    .mean()
    .sort_values(ascending=False)
)
fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(x=brand_price.values, y=brand_price.index, ax=ax)
ax.set_title("Average price by brand (top 15 by listings)")
ax.set_xlabel("Average price")
ax.set_ylabel("")
fig.tight_layout()
fig.savefig(CHART_DIR / "pricing_analysis.png", dpi=150)
plt.show()

In [ ]:
yearly = df.groupby("year")["price"].mean().reset_index()
fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=yearly, x="year", y="price", marker="o", ax=ax)
ax.set_title("Average price by model year")
fig.tight_layout()
fig.savefig(CHART_DIR / "price_by_year.png", dpi=150)
plt.show()

## Vehicle analysis

Mileage vs price and fuel/transmission mix.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=df, x="mileage", y="price", alpha=0.4, ax=ax)
ax.set_title("Price vs mileage")
fig.tight_layout()
fig.savefig(CHART_DIR / "vehicle_analysis.png", dpi=150)
plt.show()

for col in ["fuel_type", "transmission"]:
    print(df[col].value_counts(dropna=False))

## Seller analysis

Listing volume and pricing by seller type.

In [ ]:
seller = (
    df.groupby("seller_type")
    .agg(listings=("price", "size"), avg_price=("price", "mean"))
    .sort_values("listings", ascending=False)
)
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=seller.reset_index(), x="seller_type", y="avg_price", ax=ax)
ax.set_title("Average price by seller type")
fig.tight_layout()
fig.savefig(CHART_DIR / "seller_analysis.png", dpi=150)
plt.show()
seller

In [ ]:
numeric = df.select_dtypes("number")
numeric.corr()["price"].sort_values(ascending=False)

## Takeaways

- Record the strongest price drivers and any outliers.
- Mirror the key findings in `outputs/business_insights.md`.
- Use `screenshots/` for Power BI dashboard captures.